# Explorare: coridorul, cererea și modelul de bază (P1)

Notebook pentru primele săptămâni (etapele M1 și M2). Rulați celulele în ordine. Ce vedeți aici intră în nota de proiectare: cum arată cererea, cum arată rețeaua, ce face modelul de bază.

Dacă datele din pachet nu sunt încă în `data/`, celulele folosesc date sintetice și vă spun asta.

In [ ]:
import os, sys
from pathlib import Path
# notebook-ul se poate deschide din notebooks/ sau din rădăcină; lucrăm mereu din rădăcină
if not Path("config/default.yaml").exists() and Path("../config/default.yaml").exists():
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sada.config import load_config
cfg = load_config("config/default.yaml")
print("Lucrez în", Path.cwd())

## 1. Profilul de cerere

Câte vehicule pe oră intră în coridor, pe parcursul dimineții. Valoarea maximă vă spune cât de aproape de capacitate este coridorul (comparați-o, la final, cu fluxul maxim din diagrama fundamentală înmulțit cu numărul de benzi).

In [ ]:
import glob
csvs = glob.glob("data/p1_coridor/*.csv")
if csvs:
    prof = pd.read_csv(csvs[0])
    print("Fișier:", csvs[0]); display(prof.head())
    print("Ajustați numele coloanelor mai jos dacă diferă.")
    hour_col, flow_col = prof.columns[0], prof.columns[1]
else:
    print("Nu găsesc profilul din pachet; folosesc un profil SINTETIC.")
    sys.path.insert(0, "scripts")
    from make_flows import synthetic_profile
    prof = synthetic_profile(base=400, peak=1800); hour_col, flow_col = "hour", "veh_per_hour"
ax = prof.plot.bar(x=hour_col, y=flow_col, legend=False)
ax.set_xlabel("Ora"); ax.set_ylabel("Vehicule pe oră"); ax.set_title("Profilul de cerere la intrarea în coridor")
plt.show()

## 2. Rețeaua SUMO

Lungimea coridorului, numărul de benzi, intersecțiile. Comparați cu ce modelați în automatul celular (3–5 km, 2–3 benzi).

In [ ]:
net_path = Path(cfg["sumo"]["net"])
if net_path.exists():
    import sumolib
    net = sumolib.net.readNet(str(net_path))
    edges = net.getEdges()
    print(f"{len(edges)} muchii, {len(net.getNodes())} noduri, lungime totală {sum(e.getLength() for e in edges)/1000:.1f} km")
    fig, ax = plt.subplots(figsize=(6, 6))
    for e in edges:
        xs, ys = zip(*e.getShape())
        ax.plot(xs, ys, lw=0.3 + 0.6 * e.getLaneNumber(), color="0.3")
    ax.set_aspect("equal"); ax.set_title("Rețeaua SUMO (grosimea = numărul de benzi)"); plt.show()
else:
    print(f"Nu există {net_path}. Vedeți DATE.md (scripts/build_sumo_net.py).")

## 3. Modelul de bază pe inel

Merge după ce implementați `RingRoad.step` (GHID.md, pasul 1). Schimbați densitatea și p și urmăriți cum apar ambuteiajele.

In [ ]:
from sada.p1.nasch import RingRoad
density, p = 0.3, 0.3
try:
    road = RingRoad(300, density, vmax=2, p=p, rng=np.random.default_rng(1))
    for _ in range(200): road.step()
    rows = []
    for _ in range(250):
        road.step(); rows.append(road.occupancy())
    img = np.ma.masked_less(np.array(rows), 0)
    plt.figure(figsize=(6, 4)); plt.imshow(img, aspect="auto", cmap="viridis", vmin=0, vmax=2, interpolation="nearest")
    plt.colorbar(label="Viteză [celule/pas]"); plt.xlabel("Poziție [celule]"); plt.ylabel("Timp [pași]")
    plt.title(f"Spațiu-timp, densitate {density}, p = {p}"); plt.show()
except NotImplementedError as e:
    print("Încă nu e implementat:", e)

## 4. Diagrama fundamentală, rapid

Densitatea critică de aici e punctul de pornire pentru pragurile controllerului.

In [ ]:
from sada.p1.fundamental import fundamental_diagram, to_physical
try:
    fd = to_physical(fundamental_diagram(np.linspace(0.05, 0.85, 12), length=500, warmup=200, measure=200,
                                         rng=np.random.default_rng(0), vmax=2, p=0.3))
    i = fd["flow_veh_h"].idxmax()
    print(f"Flux maxim ≈ {fd.flow_veh_h[i]:.0f} veh/h/bandă la ≈ {fd.density_veh_km[i]:.0f} veh/km/bandă")
    fd.plot(x="density_veh_km", y="flow_veh_h", marker="o", legend=False)
    plt.xlabel("Densitate [veh/km/bandă]"); plt.ylabel("Flux [veh/h/bandă]"); plt.show()
except NotImplementedError as e:
    print("Încă nu e implementat:", e)

## Ce notați pentru nota de proiectare

- vârful cererii și ora lui;
- lungimea coridorului și numărul de benzi;
- densitatea critică și fluxul maxim (după ce merge modelul);
- ce scenarii de cerere alegeți (scăzută, medie, ridicată) în raport cu capacitatea.